### Install Packages

In [1]:
%pip install "redisvl>=0.13.2" nltk pandas sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.7/196.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 356.1/356.1 kB 12.9 MB/s eta 0:00:00


### Install Redis

For this tutorial you will need a running instance of Redis if you don't already have one.

#### Local Redis
Use the shell script below to download, extract, and install [Redis](https://redis.io/docs/latest/operate/oss_and_stack/install/install-stack/apt/) directly from the Redis package archive for a Linux environment.

In [2]:
# NBVAL_SKIP
%%sh
sudo apt-get install lsb-release curl gpg
curl -fsSL https://packages.redis.io/gpg | sudo gpg --dearmor -o /usr/share/keyrings/redis-archive-keyring.gpg
sudo chmod 644 /usr/share/keyrings/redis-archive-keyring.gpg
echo "deb [signed-by=/usr/share/keyrings/redis-archive-keyring.gpg] https://packages.redis.io/deb $(lsb_release -cs) main" | sudo tee /etc/apt/sources.list.d/redis.list
sudo apt-get update
sudo apt-get install redis

redis-server --version
redis-server --daemonize yes --loadmodule /usr/lib/redis/modules/redisearch.so

Reading package lists...
Building dependency tree...
Reading state information...
lsb-release is already the newest version (11.1.0ubuntu4).
lsb-release set to manually installed.
curl is already the newest version (7.81.0-1ubuntu1.21).
gpg is already the newest version (2.2.27-3ubuntu2.5).
gpg set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
deb [signed-by=/usr/share/keyrings/redis-archive-keyring.gpg] https://packages.redis.io/deb jammy main
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://packages.redis.io/deb jammy InRelease [3,854 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://cloud.r-projec

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 3.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 


In [3]:
from packaging.version import Version

from redis import __version__ as redis_version
from redisvl import __version__ as redisvl_version


if Version(redis_version) < Version("7.1.0"):
    raise RuntimeError("redis-py version must be >= 7.1.0")

if Version(redisvl_version) < Version("0.13.0"):
    raise RuntimeError("redisvl version must be >= 0.13.0")

In [4]:
import os
import warnings

warnings.filterwarnings('ignore')

# Replace values below with your own if using Redis Cloud instance
REDIS_HOST = os.getenv("REDIS_HOST", "localhost")
REDIS_PORT = os.getenv("REDIS_PORT", "6379")
REDIS_PASSWORD = os.getenv("REDIS_PASSWORD", "")

# If SSL is enabled on the endpoint, use rediss:// as the URL prefix
REDIS_URL = f"redis://:{REDIS_PASSWORD}@{REDIS_HOST}:{REDIS_PORT}"

### Create redis client, load data, generate embeddings

In [5]:
from redis import Redis
from redisvl.redis.connection import RedisConnectionFactory

client = Redis.from_url(REDIS_URL)
client.ping()

if Version(client.info()["redis_version"]) < Version("8.4.0"):
    raise RuntimeError("Redis version must be >= 8.4.0")

installed_modules = RedisConnectionFactory.get_modules(client)
if "search" not in installed_modules:
    raise RuntimeError("Redisearch module is not installed")

In [57]:
import json

with open("data_40.json", 'r') as file:
    data_json = json.load(file)

In [58]:
data_json[-1]

{'id': '39',
 'text': 'When comparing Redis as a vector database versus dedicated vector databases like Pinecone or Weaviate, Redis excels in low-latency caching but may lack advanced indexing features.'}

In [18]:
from redisvl.utils.vectorize import HFTextVectorizer
from redisvl.extensions.cache.embeddings import EmbeddingsCache


# load model for embedding our movie descriptions
model = HFTextVectorizer(
    model='sentence-transformers/all-MiniLM-L6-v2',
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [59]:
# embed movie descriptions
input_data = [
    {
        **d,
        "text_vector": model.embed(d["text"], as_buffer=True)
    } for d in data_json
]

In [60]:
input_data[-1]

{'id': '39',
 'text': 'When comparing Redis as a vector database versus dedicated vector databases like Pinecone or Weaviate, Redis excels in low-latency caching but may lack advanced indexing features.',
 'text_vector': b'&O\xff<K\xfb\xb0\xbd}\xc5D\xbd\x19\xc3V=\'g\x12=\xc9f\xa2\xbd&\xef\x84\xbd8\xf1\x9a\xbb\xf9\x7f\xd0<<\xa5\xd5;\x11\xe5\x97\xbd\x1c\xd0\x8c=.{N\xbdG\xfc3\xbc\xd7%\x05\xbc\x9fW\x96\xbd\xe7S\xc4=\x86\x19\x15=k\x93\xb6\xbc\xac$\x1e\xbd\xce\x96&\xbd(\x0f\xc7\xbc}\xbf\x12\xbd\xc4n\x86\xbb!\x813=9\xea\xaf\xbdLn?\xbc\x9e\xb3\x0c\xbd\xab\x97\x82\xbd\x15\x06\x19\xbd\xc8%"\xba\xf9\xa8\x87=w\x8b\x8b<\xe1\x05\xd3<V\xad\x0b\xbeW\xdf\x9b=\xa9!\xf0;j\xf2\xd5\xbd\xe4T\x1f\xbd\x83\xfe\xf8\xba]\x916\xbc\x91\xf6p=\xfb\xd1\xb5\xbc9\xc1\xb8=\xcb/\xb2;0\x9b%=\xbe>i\xbd\xd6pC=\xf33!\xba\x05,n\xbdk3\x9f\xbdu\x1c\x86;\x0f\x873\xbd\x8bj\xa1;\xa2\xf4@\xbd\x9eo\x13=H\x08.;\xc1\xf0b\xbd6\x84\r=V\xbc9\xbd\x19\xe5\xa0=\n\xdfj\xbd\xf9Z\xe5<\x9e\xb20\xbc\xa5\xcd\x88\xbd)\xb3\x82<r\x7f\x8b=\xe8\xb3\x1

### Define Redis index schema

Below, we build a schema that represents our data objects.

In [21]:
from redisvl.schema import IndexSchema
from redisvl.index import SearchIndex


schema = IndexSchema.from_dict({
  "index": {
    "name": "data",
    "prefix": "data",
    "storage": "hash"
  },
  "fields": [
    { "name": "id", "type": "text" },
    { "name": "text", "type": "text" },
    {
        "name": "text_vector",
        "type": "vector",
        "attrs": {
            "dims": 384,
            "distance_metric": "cosine",
            "algorithm": "hnsw",
            "datatype": "float32"
        }
    }
  ]
})


In [61]:
index = SearchIndex(schema, client, validate_on_load=True)
index.create(overwrite=True, drop=True)

### Populate index

Load movie objects into Redis

In [62]:
index.load(input_data)

['data:01KHV72B8JXHKN6QH566HEKAQ7',
 'data:01KHV72B8KNR0M9PP38YC4E8QR',
 'data:01KHV72B8KNR0M9PP38YC4E8QS',
 'data:01KHV72B8KNR0M9PP38YC4E8QT',
 'data:01KHV72B8KNR0M9PP38YC4E8QV',
 'data:01KHV72B8KNR0M9PP38YC4E8QW',
 'data:01KHV72B8KNR0M9PP38YC4E8QX',
 'data:01KHV72B8KNR0M9PP38YC4E8QY',
 'data:01KHV72B8MHF6XZV0ANNM418BG',
 'data:01KHV72B8MHF6XZV0ANNM418BH',
 'data:01KHV72B8MHF6XZV0ANNM418BJ',
 'data:01KHV72B8MHF6XZV0ANNM418BK',
 'data:01KHV72B8MHF6XZV0ANNM418BM',
 'data:01KHV72B8MHF6XZV0ANNM418BN',
 'data:01KHV72B8MHF6XZV0ANNM418BP',
 'data:01KHV72B8MHF6XZV0ANNM418BQ',
 'data:01KHV72B8MHF6XZV0ANNM418BR',
 'data:01KHV72B8MHF6XZV0ANNM418BS',
 'data:01KHV72B8MHF6XZV0ANNM418BT',
 'data:01KHV72B8NB6Q0ZBDEV6Q6FCFT',
 'data:01KHV72B8NB6Q0ZBDEV6Q6FCFV',
 'data:01KHV72B8NB6Q0ZBDEV6Q6FCFW',
 'data:01KHV72B8NB6Q0ZBDEV6Q6FCFX',
 'data:01KHV72B8NB6Q0ZBDEV6Q6FCFY',
 'data:01KHV72B8NB6Q0ZBDEV6Q6FCFZ',
 'data:01KHV72B8NB6Q0ZBDEV6Q6FCG0',
 'data:01KHV72B8NB6Q0ZBDEV6Q6FCG1',
 'data:01KHV72B8NB6Q0ZBDEV6Q

# Hybrid Retrieval

In [63]:
# Sample user query (can be changed for comparisons)
user_query = "How is Python used in scalable backend systems with Redis?"

### Choosing your text scoring function and weights
There are different ways to calculate the similarity between sets of text. Options for text scoring functions are TFIDF, TFIDF.DOCNORM, BM25STD, BM25STD.NORM, BM25STD.TANH, DISMAX, DOCSCORE, and HAMMING; the default is BM25STD and is easy to configure with the `text_scorer` parameter. Just like changing you embedding model can change your vector similarity scores, changing your text similarity measure can change your text scores.

>  For more information about supported scoring algorithms, see [the Redis documentation on scoring](https://redis.io/docs/latest/develop/ai/search-and-query/advanced-concepts/scoring/).

When combining text and vector scores using a linear combination (`combination_method="LINEAR"` in `HybridQuery` and the only option for `AggregateHybridQuery`), you can control the relative balance of these scores with tunable parameters.

The FT.HYBRID API calculates the combined score as:

```python
hybrid_score = {alpha} * text_score + {beta} * vector_similarity
```

Where `alpha` can be provided to `HybridQuery` via the `linear_alpha` and `beta` is calculated as `1 - alpha`. FT.HYBRID defaults to `alpha=0.3`.

`AggregateHybridQuery` defines the combined score in reverse as:

```python
hybrid_score = {1-alpha} * text_score + {alpha} * vector_similarity
```

Where the `alpha` parameter is configurable on the `AggregateHybridQuery` class. If not specified, it defaults to `0.7`.

Try changing the `text_scorer` and `linear_alpha` parameters in the query below to see how results may change.

In [64]:
import pandas as pd

from redisvl.query.hybrid import HybridQuery

In [65]:
pd.set_option('display.max_colwidth', None)

In [66]:
vector = model.embed(user_query, as_buffer=True)

In [67]:
tfidf_query = HybridQuery(
    text=user_query,
    text_field_name="text",
    vector=vector,
    vector_field_name="text_vector",
    text_scorer="BM25", # can be one of [TFIDF, TFIDF.DOCNORM, BM25, DISMAX, DOCSCORE, BM25STD]
    stopwords=None,
	combination_method="LINEAR",
    linear_alpha=0.6, # weight the text score higher
    return_fields=["id", "text"],
	yield_text_score_as="text_score",
    yield_vsim_score_as="vector_similarity",
    yield_combined_score_as="hybrid_score",
)

In [68]:
results = index.query(tfidf_query)
pd.DataFrame(results[:3])

,text_score,id,text,vector_similarity,hybrid_score
0,0.621931239358,39,"When comparing Redis as a vector database versus dedicated vector databases like Pinecone or Weaviate, Redis excels in low-latency caching but may lack advanced indexing features.",0.785794645548,0.687476601834
1,0.621931239358,32,Redis as a vector store is different from Redis as a cache. As a vector store it stores high-dimensional embeddings; as a cache it stores key-value pairs with TTL expiry.,0.78437897563,0.686910333867
2,0.505767505998,15,Amazon provides cloud services including managed Redis deployments for scalable applications.,0.817494899035,0.630458463212


In [70]:
from google import genai
from google.genai import types
from google.colab import userdata

In [71]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", userdata.get('AI_STUDIO_KEY'))
MODEL_ID = "gemini-3-flash-preview"

In [72]:
client = genai.Client(api_key=GOOGLE_API_KEY)

In [69]:
def traditional_rag_answer(client: genai.Client,
                           query: str, chunks: list[dict]) -> str:

    """Feed retrieved chunks to Gemini and get an answer — no agent loop."""
    context = "\n\n".join([f"[Chunk {c['id']}] {c['text']}" for c in chunks])
    prompt  = (
        f"Answer the question using ONLY the context below. "
        f"If the context is insufficient say 'I cannot answer from the given context.'\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}"
    )
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
    )
    return response.text

In [77]:
print(traditional_rag_answer(client, user_query, results[:3]))

I cannot answer from the given context.


# Agentic RAG using Google ADK

In [45]:
import os
import json
from redisvl.query import VectorQuery

In [46]:
def semantic_search(query: str, top_k: int = 3) -> str:
    """
    Search the knowledge base using semantic (vector) similarity.
    Use this to find documents that are conceptually related to the query,
    even if they don't share the exact same words.
    Returns document IDs and their text content.
    """
    vector = model.embed(query, as_buffer=True)
    vq = VectorQuery(
        vector=vector,
        vector_field_name="text_vector",
        return_fields=["id", "text"],
        num_results=top_k,
    )
    results = index.query(vq)
    if not results:
        return "No results found."
    return "\n\n".join([f"[ID:{r['id']}] {r['text']}" for r in results])

In [47]:
def hybrid_search(query: str, alpha: float = 0.5, top_k: int = 3) -> str:
    """
    Search using a hybrid of BM25 keyword scoring + vector similarity.
    Use this when semantic_search returns off-topic results, or when
    the query contains important specific keywords (e.g. a product name,
    acronym, or technical term) that must appear in the result.
    alpha: weight for text score (0=pure vector, 1=pure keyword).
    """
    vector = model.embed(query, as_buffer=True)
    hq = HybridQuery(
        text=query,
        text_field_name="text",
        vector=vector,
        vector_field_name="text_vector",
        text_scorer="BM25",
        stopwords=None,
        combination_method="LINEAR",
        linear_alpha=alpha,
        return_fields=["id", "text"],
        yield_combined_score_as="hybrid_score",
    )
    results = index.query(hq)
    if not results:
        return "No results found."
    return "\n\n".join([
        f"[ID:{r['id']} | hybrid_score:{r.get('hybrid_score', 'N/A')}] {r['text']}"
        for r in results[:top_k]
    ])

In [48]:
def keyword_filter(keyword: str) -> str:
    """
    Find all documents that contain a specific keyword (exact match, case-insensitive).
    Use this to locate documents about a very specific concept or term
    that might be diluted in semantic search results.
    """
    keyword_lower = keyword.lower()
    # Full-text search via RediSearch TAG/TEXT field
    try:
        from redis.commands.search.query import Query as RawQuery
        raw_client  = index.client
        raw_query   = RawQuery(f"@text:{{{keyword_lower}}}").return_fields("id", "text").paging(0, 10)
        raw_results = raw_client.ft("data").search(raw_query)
        docs = raw_results.docs
        if not docs:
            raise ValueError("no results via TAG search")
        return "\n\n".join([f"[ID:{d.id}] {d.text}" for d in docs])
    except Exception:
        # Fallback: scan all data in-memory (works for small datasets like this demo)
        from redis import Redis
        r = Redis.from_url(REDIS_URL, decode_responses=True)
        keys = r.keys("data:*")
        matches = []
        for key in keys:
            text = r.hget(key, "text") or ""
            if keyword_lower in text.lower():
                doc_id = r.hget(key, "id") or key
                matches.append(f"[ID:{doc_id}] {text}")
        return "\n\n".join(matches) if matches else f"No documents found containing '{keyword}'."


In [49]:
def get_document(doc_id: str) -> str:
    """
    Retrieve the full text of a specific document by its ID.
    Use this when you have identified a promising document ID from a prior
    search and want to read its complete content before forming an answer.
    """
    from redis import Redis
    r = Redis.from_url(REDIS_URL, decode_responses=True)
    text = r.hget(f"data:{doc_id}", "text")
    if text:
        return f"[ID:{doc_id}] {text}"
    return f"Document ID '{doc_id}' not found."


In [50]:
TOOLS = types.Tool(
    function_declarations=[

        types.FunctionDeclaration(
            name="semantic_search",
            description=(
                "Search the knowledge base using semantic/vector similarity. "
                "Best for concept-level or paraphrased queries."
            ),
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "query": types.Schema(type=types.Type.STRING,
                                         description="Natural language search query."),
                    "top_k": types.Schema(type=types.Type.INTEGER,
                                          description="Number of results to return (default 3)."),
                },
                required=["query"],
            ),
        ),

        types.FunctionDeclaration(
            name="hybrid_search",
            description=(
                "Search using BM25 keyword scoring combined with vector similarity. "
                "Use when the query contains critical keywords that must match exactly, "
                "or when semantic_search misses relevant results."
            ),
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "query": types.Schema(type=types.Type.STRING,
                                          description="The search query string."),
                    "alpha": types.Schema(type=types.Type.NUMBER,
                                          description="Keyword weight: 0.0=pure vector, 1.0=pure keyword. Default 0.5."),
                    "top_k": types.Schema(type=types.Type.INTEGER,
                                          description="Number of results to return (default 3)."),
                },
                required=["query"],
            ),
        ),

        types.FunctionDeclaration(
            name="keyword_filter",
            description=(
                "Find all documents containing a specific keyword. "
                "Use for precise terms like acronyms, product names, or technical jargon."
            ),
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "keyword": types.Schema(type=types.Type.STRING,
                                            description="The exact keyword to search for."),
                },
                required=["keyword"],
            ),
        ),

        types.FunctionDeclaration(
            name="get_document",
            description="Retrieve the full content of a specific document by its numeric ID.",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "doc_id": types.Schema(type=types.Type.STRING,
                                           description="Document ID as a string, e.g. '5'."),
                },
                required=["doc_id"],
            ),
        ),
    ]
)

In [51]:
TOOL_MAP = {
    "semantic_search": semantic_search,
    "hybrid_search":   hybrid_search,
    "keyword_filter":  keyword_filter,
    "get_document":    get_document,
}

In [52]:
SYSTEM_PROMPT = """You are a precise research assistant with access to a knowledge base.
Answer questions ONLY using evidence retrieved via your tools.

YOUR STRATEGY:
1. Start with semantic_search to get an initial set of relevant documents.
2. If results seem off or incomplete, try hybrid_search with a refined query.
3. Use keyword_filter to hunt for documents containing a critical specific term.
4. Use get_document to read a specific document more carefully when needed.
5. You may call tools MULTIPLE TIMES — do not stop at the first result.
6. Cross-reference: if a question asks about TWO things, search for EACH separately.
7. Only write your final answer when you are confident you have sufficient evidence.
8. Cite which document IDs support each claim in your answer.

If the knowledge base genuinely does not contain the answer, say so clearly."""


In [53]:
def agentic_rag(query: str, verbose: bool = True) -> str:
    """
    Run the Agentic RAG loop.

    Parameters
    ----------
    query   : The user's question.
    verbose : If True, prints each tool call and result (great for demos).

    Returns
    -------
    The agent's final grounded answer as a string.
    """
    client   = genai.Client(api_key=GOOGLE_API_KEY)
    messages = [types.Content(role="user", parts=[types.Part(text=query)])]

    if verbose:
        print(f"\n{'═'*68}")
        print(f"  AGENTIC RAG  |  Query: {query}")
        print(f"{'═'*68}")

    for iteration in range(10):  # safety cap at 10 tool-call rounds
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=messages,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
                tools=[TOOLS],
                temperature=0.1,
            ),
        )

        all_parts     = response.candidates[0].content.parts
        fn_call_parts = [p for p in all_parts if p.function_call is not None]
        text_parts    = [p for p in all_parts if p.text]

        # ── No more tool calls → agent is done ───────────────────────────────
        if not fn_call_parts:
            answer = "\n".join(p.text for p in text_parts if p.text)
            if verbose:
                print(f"\n✅ Final Answer (after {iteration} tool-call round(s)):\n")
                print(answer)
            return answer

        # ── Append model turn (with function calls) to history ───────────────
        messages.append(types.Content(role="model", parts=all_parts))

        # ── Execute each tool call ────────────────────────────────────────────
        tool_response_parts = []
        for part in fn_call_parts:
            fn_name = part.function_call.name
            fn_args = dict(part.function_call.args)

            if verbose:
                print(f"\n  🔧 [{iteration+1}] {fn_name}({json.dumps(fn_args, indent=None)})")

            result = TOOL_MAP[fn_name](**fn_args) if fn_name in TOOL_MAP else f"Unknown tool: {fn_name}"

            if verbose:
                preview = result[:300] + ("…" if len(result) > 300 else "")
                print(f"     ↳ {preview}")

            tool_response_parts.append(
                types.Part(
                    function_response=types.FunctionResponse(
                        name=fn_name,
                        response={"result": result},
                    )
                )
            )

        # ── Feed tool results back to the agent ───────────────────────────────
        messages.append(types.Content(role="user", parts=tool_response_parts))

    return "Agent exceeded max iterations without a conclusive answer."


In [78]:
# Sample user query (can be changed for comparisons)
user_query = "How is Python used in scalable backend systems with Redis?"

In [79]:
answer = agentic_rag(user_query, verbose=True)


════════════════════════════════════════════════════════════════════
  AGENTIC RAG  |  Query: How is Python used in scalable backend systems with Redis?
════════════════════════════════════════════════════════════════════

  🔧 [1] semantic_search({"query": "Python Redis scalable backend systems"})
     ↳ [ID:data:01KHV72B8NB6Q0ZBDEV6Q6FCFV] Redis is an in-memory data structure store that can be used as a database, cache, and message broker. It supports data structures such as strings, hashes, lists, and sets.

[ID:data:01KHV72B8JXHKN6QH566HEKAQ7] Python is a high-level programming language used for …

  🔧 [2] hybrid_search({"query": "Python Redis scalability message broker task queue"})
     ↳ [ID:39 | hybrid_score:0.667027689214] When comparing Redis as a vector database versus dedicated vector databases like Pinecone or Weaviate, Redis excels in low-latency caching but may lack advanced indexing features.

[ID:32 | hybrid_score:0.666096172925] Redis as a vector store is different fr